# TEMP — Synthetic Floor GPU camera regression test

این نوت‌بوک عمداً کنار نوت‌بوک اصلی ساخته شده و به قبلی دست نمی‌زند.

هدفش فقط تست سریع/قابل‌تکرار خروجی دوربین است:
- هر بار repo را fresh clone می‌کند.
- می‌تواند `main`، یک branch، یا head یک Pull Request را تست کند.
- تنظیمات reconstruction را explicit می‌گذارد: `1800 frames @ 30fps`.
- بعد از رندر، ویدیو، storyboard و camera diagnostics می‌دهد.

برای تغییرات بعدی لازم نیست خود notebook را edit کنی؛ فقط فرم سلول اول را عوض کن یا دوباره همان PR/branch را run بگیر.


In [ ]:
#@title 1) Test configuration
# Repository / ref to test.
REPO_URL = "https://github.com/sayyedalimrj/MyCon.git"  #@param {type:"string"}
REF_MODE = "pr"  #@param ["pr", "branch", "main"]
PR_NUMBER = 28  #@param {type:"integer"}
BRANCH = "fix/restore-good-blender-camera"  #@param {type:"string"}

# Isolated checkout. This notebook deletes/recreates it when RESET_WORKDIR=True.
WORKDIR = "/content/mycon_temp_camera_test"  #@param {type:"string"}
RESET_WORKDIR = True  #@param {type:"boolean"}

# Render settings. Keep these explicit so we never accidentally test a 4s/debug export.
STAGE_ONLY = 7  #@param {type:"integer"}
PRESET = "balanced"  #@param ["debug", "balanced", "hq"]
FRAMES = 1800  #@param {type:"integer"}
FPS = 30  #@param {type:"integer"}
RESOLUTION_W = 960  #@param {type:"integer"}
RESOLUTION_H = 540  #@param {type:"integer"}
SAMPLES = 96  #@param {type:"integer"}
WORKERS = 1  #@param {type:"integer"}

# Drive persistence is optional for this temp test.
MOUNT_DRIVE = False  #@param {type:"boolean"}
DRIVE_ROOT = ""  #@param {type:"string"}

# Runtime switches.
INSTALL_BLENDER_IF_MISSING = True  #@param {type:"boolean"}
RUN_RENDER = True  #@param {type:"boolean"}
STRICT_RENDER = True  #@param {type:"boolean"}
SAVE_BLEND = False  #@param {type:"boolean"}

print("Configured temp camera test.")


In [ ]:
#@title 2) Fresh clone and checkout selected ref
import os
import shutil
import subprocess
from pathlib import Path

workdir = Path(WORKDIR)
repo_dir = workdir / "MyCon"

def run(cmd, cwd=None, check=True, env=None):
    print("\n$ " + " ".join(map(str, cmd)))
    p = subprocess.run(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
    print(p.stdout)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed with rc={p.returncode}: {' '.join(map(str, cmd))}")
    return p

if RESET_WORKDIR and workdir.exists():
    shutil.rmtree(workdir)
workdir.mkdir(parents=True, exist_ok=True)

if not repo_dir.exists():
    run(["git", "clone", REPO_URL, str(repo_dir)])

run(["git", "fetch", "--all", "--prune"], cwd=repo_dir)

if REF_MODE == "pr":
    test_ref = f"temp-pr-{PR_NUMBER}"
    run(["git", "fetch", "origin", f"pull/{PR_NUMBER}/head:{test_ref}"], cwd=repo_dir)
    run(["git", "checkout", test_ref], cwd=repo_dir)
elif REF_MODE == "branch":
    run(["git", "fetch", "origin", BRANCH], cwd=repo_dir)
    run(["git", "checkout", "-B", BRANCH, f"origin/{BRANCH}"], cwd=repo_dir)
else:
    run(["git", "checkout", "main"], cwd=repo_dir)
    run(["git", "pull", "--ff-only", "origin", "main"], cwd=repo_dir)

sha = run(["git", "rev-parse", "HEAD"], cwd=repo_dir).stdout.strip()
print(f"\nTesting commit: {sha}")


In [ ]:
#@title 3) Install/verify Blender and Python dependencies
from pathlib import Path
import subprocess, sys, os

repo_root = Path(WORKDIR) / "MyCon" / "repository"
example_root = repo_root / "examples" / "synthetic_floor_7stage"
blender_path = Path("/content/blender/blender")

if INSTALL_BLENDER_IF_MISSING and not blender_path.exists():
    setup_script = example_root / "scripts" / "setup_colab_blender.sh"
    if setup_script.exists():
        run(["bash", str(setup_script)], cwd=repo_root)
    else:
        raise FileNotFoundError(f"Blender setup script not found: {setup_script}")

if not blender_path.exists():
    print("WARNING: /content/blender/blender not found. Set INSTALL_BLENDER_IF_MISSING=True or pass --blender manually.")
else:
    run([str(blender_path), "--version"], check=False)

# Lightweight Python deps used by the orchestrator/diagnostics.
run([sys.executable, "-m", "pip", "install", "-q", "numpy", "pillow", "imageio", "imageio-ffmpeg", "matplotlib", "pyyaml", "tqdm"], check=True)
print("Environment ready.")


In [ ]:
#@title 4) Run the GPU render with explicit reconstruction settings
from pathlib import Path
import os, sys, subprocess, json

repo_root = Path(WORKDIR) / "MyCon" / "repository"
example_root = repo_root / "examples" / "synthetic_floor_7stage"
src = example_root / "src"
script = example_root / "scripts" / "run_blender_gpu.py"

cmd = [
    sys.executable, str(script),
    "--blender", "/content/blender/blender",
    "--preset", PRESET,
    "--stage-only", str(STAGE_ONLY),
    "--frames", str(FRAMES),
    "--fps", str(FPS),
    "--resolution", str(RESOLUTION_W), str(RESOLUTION_H),
    "--samples", str(SAMPLES),
    "--workers", str(WORKERS),
    "--force",
]

if STRICT_RENDER:
    cmd.append("--strict-render")
if SAVE_BLEND:
    cmd.append("--save-blend")
if MOUNT_DRIVE:
    cmd.append("--mount-drive")
if DRIVE_ROOT.strip():
    cmd += ["--drive-root", DRIVE_ROOT.strip()]

env = os.environ.copy()
env["PYTHONPATH"] = str(src) + os.pathsep + env.get("PYTHONPATH", "")

print("Render command:")
print(" ".join(cmd))

if RUN_RENDER:
    p = subprocess.run(cmd, cwd=repo_root, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)
    if p.returncode != 0:
        raise RuntimeError(f"Render failed with rc={p.returncode}")
else:
    print("RUN_RENDER=False, skipped render.")


In [ ]:
#@title 5) Locate outputs and verify this is really a 60s / 1800-frame export
from pathlib import Path
import sys, os, json, glob, subprocess
import imageio.v3 as iio
from PIL import Image
import numpy as np

repo_root = Path(WORKDIR) / "MyCon" / "repository"
example_root = repo_root / "examples" / "synthetic_floor_7stage"
src = example_root / "src"
sys.path.insert(0, str(src))

from synthetic_floor.scene_spec import load_scene_spec
from synthetic_floor import paths as P

spec = load_scene_spec(example_root / "config" / "scene.yaml")
stage_dir = P.stage_blender_render_dir(spec, STAGE_ONLY)
sub = P.stage_blender_subdirs(spec, STAGE_ONLY)
video_path = P.stage_video_path(spec, STAGE_ONLY, gpu=True)
rgb_dir = sub["rgb"]
camera_path_json = sub["camera_path"]
camera_poses_json = stage_dir / "camera_poses.json"

print("Stage dir:", stage_dir)
print("RGB dir:", rgb_dir)
print("Video:", video_path)
print("camera_path.json:", camera_path_json)
print("camera_poses.json:", camera_poses_json)

pngs = sorted(rgb_dir.glob("frame_*.png"))
print(f"RGB frames on disk: {len(pngs)}")
if len(pngs) != FRAMES:
    raise AssertionError(f"Expected {FRAMES} RGB frames, found {len(pngs)}")

if not video_path.exists():
    raise FileNotFoundError(f"Video not found: {video_path}")

# imageio metadata is sometimes backend-dependent; frame count from PNGs is the source of truth.
print(f"OK: found {len(pngs)} frames for {FRAMES/FPS:.1f}s @ {FPS}fps")


In [ ]:
#@title 6) Camera diagnostics: height, pitch, speed, and bad-view checks
from pathlib import Path
import json, math
import numpy as np

def load_camera_records():
    if camera_path_json.exists():
        data = json.loads(camera_path_json.read_text())
        frames = data.get("frames", [])
        eyes = np.array([fr.get("eye_world_m", fr.get("eye")) for fr in frames], dtype=float)
        targets = np.array([fr.get("target_world_m", fr.get("target")) for fr in frames], dtype=float)
        return eyes, targets, "camera_path.json"
    if camera_poses_json.exists():
        data = json.loads(camera_poses_json.read_text())
        frames = data.get("frames", [])
        eyes = np.array([fr["eye"] for fr in frames], dtype=float)
        targets = np.array([fr["target"] for fr in frames], dtype=float)
        return eyes, targets, "camera_poses.json"
    raise FileNotFoundError("No camera path JSON found.")

eyes, targets, source = load_camera_records()
d = targets - eyes
norm = np.linalg.norm(d, axis=1)
pitch = np.degrees(np.arcsin(np.clip(d[:, 2] / np.maximum(norm, 1e-9), -1, 1)))
step = np.linalg.norm(np.diff(eyes[:, :2], axis=0), axis=1)
speed = step * FPS

print("Camera source:", source)
print("Frames:", len(eyes))
print("Eye Z min/median/max:", float(eyes[:,2].min()), float(np.median(eyes[:,2])), float(eyes[:,2].max()))
print("Pitch min/median/max:", float(pitch.min()), float(np.median(pitch)), float(pitch.max()))
print("Speed m/s p50/p95/max:", float(np.percentile(speed, 50)), float(np.percentile(speed, 95)), float(speed.max()))

# Conservative checks for the old-good style production camera.
errors = []
if len(eyes) != FRAMES:
    errors.append(f"camera frame count {len(eyes)} != {FRAMES}")
if eyes[:,2].max() > 2.25:
    errors.append("camera eye height got too high")
if pitch.max() > 35:
    errors.append("camera pitched too far upward; likely roof/sky issue")
if pitch.min() < -35:
    errors.append("camera pitched too far downward")
if np.percentile(speed, 95) > 2.2:
    errors.append("camera speed too high for stable reconstruction overlap")

if errors:
    raise AssertionError("\n".join(errors))
print("OK: camera diagnostics look stable.")


In [ ]:
#@title 7) Create and display a storyboard from the rendered video
from pathlib import Path
from PIL import Image, ImageDraw
import numpy as np
import math
from IPython.display import display

pngs = sorted(rgb_dir.glob("frame_*.png"))
n_samples = min(24, len(pngs))
indices = np.linspace(0, len(pngs) - 1, n_samples, dtype=int)

thumb_w = 240
thumbs = []
for idx in indices:
    img = Image.open(pngs[idx]).convert("RGB")
    scale = thumb_w / img.width
    thumb_h = int(round(img.height * scale))
    img = img.resize((thumb_w, thumb_h), Image.LANCZOS)
    draw = ImageDraw.Draw(img)
    draw.rectangle((0, 0, 95, 20), fill=(0, 0, 0))
    draw.text((5, 4), f"f={idx+1}", fill=(255, 255, 255))
    thumbs.append(img)

cols = 4
rows = int(math.ceil(len(thumbs) / cols))
w = cols * thumb_w
h = rows * thumbs[0].height
sheet = Image.new("RGB", (w, h), "white")
for i, img in enumerate(thumbs):
    x = (i % cols) * thumb_w
    y = (i // cols) * img.height
    sheet.paste(img, (x, y))

storyboard_path = stage_dir / f"stage_{STAGE_ONLY:02d}_temp_storyboard.jpg"
sheet.save(storyboard_path, quality=92)
print("Storyboard:", storyboard_path)
display(sheet)


In [ ]:
#@title 8) Final summary and files to inspect
print("Tested ref:")
run(["git", "rev-parse", "--short", "HEAD"], cwd=repo_dir)

print("\nImportant outputs:")
print("Video:      ", video_path)
print("Storyboard: ", storyboard_path)
print("Stage dir:  ", stage_dir)
print("Log:        ", stage_dir / "blender_render.log")
print("\nIf this looks good, merge the PR/branch you tested. If it looks bad, upload the storyboard/video and rerun after the next patch without editing this notebook.")
